In [3]:
import torch
import torch.nn.functional as F

batch_size = 3
input_dim = 4
hidden_dim = 6
output_dim = 1
torch.manual_seed(44) # 确保可复现
X = torch.randn(batch_size, input_dim)
Y = torch.randint(low=0, high=2, size=(batch_size, output_dim)).float()

# initial weight, one for manual, one for pytorch
W1 = torch.randn(input_dim, hidden_dim)
b1 = torch.randn(hidden_dim)
W2 = torch.randn(hidden_dim, output_dim)
b2 = torch.randn(output_dim)
# pytorch autograd
W1_auto = W1.clone().detach().requires_grad_(True)
b1_auto = b1.clone().detach().requires_grad_(True)
W2_auto = W2.clone().detach().requires_grad_(True)
b2_auto = b2.clone().detach().requires_grad_(True)
# A: manual forward + backward
# forward pass
Z1 = X.mm(W1) + b1
H = F.relu(Z1)
Z2 = H.mm(W2) + b2
Y_hat = torch.sigmoid(Z2)
# loss compute BCE loss
loss_manual = -(Y * torch.log(Y_hat) + (1 - Y) * torch.log(1 - Y_hat)).mean()
# backward pass
dZ2 = (Y_hat - Y) / batch_size # for .mean()
dW2_manual = H.t().mm(dZ2)
db2_manual = dZ2.sum(dim=0)
dH = dZ2.mm(W2.t())
dZ1 = dH * (Z1 > 0).float()
dW1_manual = X.t().mm(dZ1)
db1_manual = dZ1.sum(dim=0)

# B: autograd loss.backward()
Z1_auto = X.mm(W1_auto) + b1_auto
H_auto = F.relu(Z1_auto)
Z2_auto = H_auto.mm(W2_auto) + b2_auto
Y_hat_auto = torch.sigmoid(Z2_auto)
loss_auto = F.binary_cross_entropy(Y_hat_auto, Y)
# magical 
loss_auto.backward()

def compare_grads(name, manual_grad, auto_grad):
    is_close = torch.allclose(manual_grad, auto_grad, atol=1e-7)
    print(f"{name} grad is close: {is_close}")
    if not is_close:
        print(f"manual_grad: {manual_grad} ")
        print(f"auto_grad: {auto_grad}")

print(f"Loss is the same: {torch.allclose(loss_manual, loss_auto)} \n")
compare_grads("W2", dW2_manual, W2_auto.grad)
compare_grads("b2", db2_manual, b2_auto.grad)
compare_grads("W1", dW1_manual, W1_auto.grad)
compare_grads("b1", db1_manual, b1_auto.grad)



Loss is the same: True 

W2 grad is close: True
b2 grad is close: True
W1 grad is close: True
b1 grad is close: True
